# Discount Predictor — Simple Linear Regression
### IIT-style guided lab | Python + Machine Learning | Google Colab

**Goal:** predict a product's discount percentage from its original price using a simple linear regression model.

> **Teaching note:** This project creates a synthetic dataset for learning. The relationship is intentionally simple and is not real business data. A model trained on synthetic data should not be used for actual pricing decisions.

## Learning outcomes
1. Create and inspect a dataset with Python, lists, dictionaries, pandas, and NumPy.
2. Understand features (`X`) and target (`y`), then split data into training and testing sets.
3. Fit simple linear regression, interpret slope/intercept, and calculate R².
4. Visualize training/testing data and the regression line.
5. Make a prediction through a small interactive input widget.
6. Learn the Python concepts used in each section and prepare the project for GitHub.

**How to use:** Open this notebook in Google Colab and run cells from top to bottom.


## 1. Setup and imports

**Python concepts:** comments, importing modules, aliases, function calls.

- `numpy` helps create numeric data.
- `pandas` stores data in a table called a DataFrame.
- `matplotlib` draws charts.
- `scikit-learn` provides the model and evaluation tools.


In [ ]:
# If running in Google Colab, these libraries are usually preinstalled.
# Uncomment the next line only if you need to install them.
# %pip install numpy pandas matplotlib scikit-learn ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("Libraries imported successfully.")


## 2. Create a synthetic dataset

We generate original prices and a discount percentage. The target follows an approximate linear relationship with some random noise.

**Python concepts:** variables, assignment, function arguments, random numbers, arrays, arithmetic operations, reproducibility with a seed.


In [ ]:
# A fixed seed makes the generated dataset reproducible.
np.random.seed(42)

n_samples = 200
original_price = np.random.uniform(100, 5000, n_samples)
noise = np.random.normal(loc=0, scale=4, size=n_samples)

# Synthetic teaching rule: higher-priced items tend to receive slightly higher discounts.
discount_percent = 8 + (original_price * 0.006) + noise
discount_percent = np.clip(discount_percent, 1, 50)  # Keep values in a plausible range.

# Build a table from a dictionary of arrays.
df = pd.DataFrame({
    "Original_Price": original_price.round(2),
    "Discount_Percent": discount_percent.round(2)
})

df.head()


## 3. Inspect and understand the data

**Python concepts:** method calls, attributes, indexing, descriptive statistics, chained operations.


In [ ]:
print("Dataset shape (rows, columns):", df.shape)
print("\\nColumn data types:")
print(df.dtypes)
print("\\nMissing values per column:")
print(df.isna().sum())
print("\\nSummary statistics:")
display(df.describe())


## 4. Visualize the relationship

**Python concepts:** plotting, function arguments, labels, title, grid, figure size.


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Original_Price"], df["Discount_Percent"], alpha=0.75)
plt.xlabel("Original price")
plt.ylabel("Discount (%)")
plt.title("Original Price vs Discount Percentage")
plt.grid(True, alpha=0.3)
plt.show()


## 5. Define feature (X) and target (y)

The feature is the input used for prediction. The target is what we want the model to predict. Scikit-learn expects `X` to be two-dimensional, so we use double brackets for the feature column.

**Python concepts:** variables, DataFrame column selection, 2D vs 1D data.


In [ ]:
X = df[["Original_Price"]]
y = df["Discount_Percent"]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 6. Split training and testing data

Training data teaches the model. Testing data gives an estimate of performance on examples it did not train on.

**Python concepts:** function calls, keyword arguments, tuple unpacking, train/test split.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 7. Train the simple linear regression model

A simple linear regression predicts using:

\[
\hat{y} = b_0 + b_1x
\]

- `b0` (intercept): predicted discount when price is zero.
- `b1` (slope): expected change in predicted discount for each one-unit increase in price.

**Python concepts:** creating an object, calling methods, attributes, model fitting.


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

intercept = model.intercept_
slope = model.coef_[0]

print(f"Intercept (b0): {intercept:.4f}")
print(f"Slope (b1): {slope:.6f}")
print(f"Equation: predicted discount = {intercept:.4f} + ({slope:.6f} × price)")


## 8. Test the model and evaluate it

R² measures how much of the variation in the target is explained by the model on this test set. MAE and RMSE describe prediction error in discount percentage points. A negative R² is possible and indicates poor test performance.

**Python concepts:** prediction, function calls, metrics, formatted strings.


In [ ]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Test R²: {r2:.3f}")
print(f"Test MAE: {mae:.2f} percentage points")
print(f"Test RMSE: {rmse:.2f} percentage points")


## 9. Plot training data, testing data, and regression line

**Python concepts:** sorting, NumPy arrays, plotting multiple series, legend.


In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(X_train["Original_Price"], y_train, label="Training data", alpha=0.65)
plt.scatter(X_test["Original_Price"], y_test, label="Testing data", alpha=0.8)

price_line = np.linspace(df["Original_Price"].min(), df["Original_Price"].max(), 200)
discount_line = model.predict(pd.DataFrame({"Original_Price": price_line}))

plt.plot(price_line, discount_line, label="Regression line", linewidth=2)
plt.xlabel("Original price")
plt.ylabel("Discount (%)")
plt.title("Simple Linear Regression: Training, Testing, and Fit")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


## 10. Make a prediction for a new price

The model predicts the average pattern it learned; it does not guarantee a real-world discount. The clipping below is only a display guard for this synthetic example.

**Python concepts:** user input, type conversion, conditionals, exception handling, functions.


In [ ]:
def predict_discount(price):
    """Return the model's estimated discount percentage for a given price."""
    if price <= 0:
        raise ValueError("Price must be greater than zero.")

    input_frame = pd.DataFrame({"Original_Price": [price]})
    prediction = model.predict(input_frame)[0]
    return float(np.clip(prediction, 0, 100))

# Try changing this value.
example_price = 1500
print(f"Estimated discount for price ₹{example_price:,.2f}: {predict_discount(example_price):.2f}%")


### Optional: interactive prediction widget (Google Colab)

If the slider does not render in your environment, use the function in the previous cell.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    price_slider = widgets.FloatSlider(
        value=1500, min=100, max=5000, step=50,
        description="Price ₹", continuous_update=False
    )
    output = widgets.Output()

    def update_prediction(change):
        with output:
            output.clear_output()
            print(f"Estimated discount: {predict_discount(change['new']):.2f}%")

    price_slider.observe(update_prediction, names="value")
    display(price_slider, output)
    print(f"Initial estimate: {predict_discount(price_slider.value):.2f}%")
except ImportError:
    print("ipywidgets is not available. Use predict_discount(price) instead.")


## 11. Export the generated dataset

This saves a CSV file in the current Colab session. It is generated by the notebook, so you do not need to download a separate dataset.

**Python concepts:** method calls, file paths, keyword arguments.


In [ ]:
csv_path = "discount_dataset.csv"
df.to_csv(csv_path, index=False)
print(f"Saved dataset to: {csv_path}")

# In Google Colab, uncomment to download it to your computer:
# from google.colab import files
# files.download(csv_path)


## 12. Python concept checklist

| Concept | Where it appears | Why it matters |
|---|---|---|
| Variables and assignment | Dataset and model sections | Store values and objects |
| Imports and aliases | Setup | Reuse libraries |
| Lists/dictionaries and arrays | Dataset creation | Organize and compute data |
| Functions and arguments | `np.random`, `train_test_split`, `predict_discount` | Reuse operations |
| Attributes and methods | `df.head()`, `model.fit()` | Work with objects |
| Indexing and column selection | `df[["Original_Price"]]`, `coef_[0]` | Select relevant data |
| Arithmetic and comparisons | Synthetic formula and validation | Transform and check values |
| `if` and `raise` | Prediction function | Validate inputs |
| `try` / `except` | Optional widget | Handle missing optional dependency |
| f-strings | Output messages | Format results |
| Train/test split | Section 6 | Evaluate unseen examples |
| Model fitting and prediction | Sections 7–8 | Learn and apply a model |
| Visualization | Sections 4 and 9 | Inspect relationships and model fit |
| CSV export | Section 11 | Save generated data |

## 13. Questions to test your understanding

1. What is the difference between a feature and a target?
2. Why do we keep a test set separate from training data?
3. What does the slope mean in this model?
4. Why is R² not enough by itself to evaluate every model?
5. What might go wrong if the relationship between price and discount is not linear?
6. Why should synthetic data not be treated as evidence about real retail discounts?

## 14. Next steps

- Replace the synthetic data with a real, permissioned dataset.
- Check outliers and data quality before training.
- Compare with a baseline model and other regression methods.
- Avoid extrapolating beyond the range represented by the training data.
- Record the dataset source, assumptions, and limitations in the GitHub README.
